# 03. BigSet Quality Assessment and Feature Freeze

This notebook assesses the three BigSet supplements and freezes the feature list (the locked set of predictors) and the security-control coefficients \(\alpha_k\) (effectiveness per dollar for each control family) before any model is fit.

BigSet rows are compiled from publicly posted pages. They are a convenience sample of publicly posted pages (Lundman, 2003; Baranauskas, 2020). News can shift emphasis among the four CPTED control families (Cozens et al., 2005); the three locked predictors are retained.

The blocked place split is declared here, before themes are counted. Random cross-validation leaks spatial dependence (Liu et al., 2022; Varoquaux & Colliot, 2023). We hold out two Census divisions and exclude those divisions from feature selection.

In [21]:
from __future__ import annotations

import re
from collections import Counter
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

HERE = Path.cwd()
if (HERE / "data" / "processed" / "place_year_clean.csv").exists():
    ROOT = HERE
elif (HERE.parent / "data" / "processed" / "place_year_clean.csv").exists():
    ROOT = HERE.parent
else:
    raise FileNotFoundError("Run this notebook from the Capstone root or from scripts/.")

DATA = ROOT / "data"
PROCESSED = DATA / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

qa_log: list[dict] = []


def log_qa(**kwargs) -> None:
    qa_log.append(kwargs)


print("ROOT", ROOT)


ROOT c:\Users\Owner\OneDrive\Capstone


## File Gate

Assessment begins only when the cleaned official table and the three BigSet role CSVs are on disk.

In [22]:
CLEAN = PROCESSED / "place_year_clean.csv"
GAPS = DATA / "cde_gaps" / "bigset_cde_gaps.csv"
COST = DATA / "control_cost" / "bigset_security_prices.csv"
NEWS = DATA / "feature_selection_corpus" / "bigset_property_crime_news.csv"
required = [CLEAN, GAPS, COST, NEWS]
missing = [p for p in required if not p.exists()]
print("missing", missing)
assert not missing, missing

clean = pd.read_csv(CLEAN)
gaps = pd.read_csv(GAPS)
cost = pd.read_csv(COST)
news = pd.read_csv(NEWS)
print("clean", len(clean), "gaps", len(gaps), "cost", len(cost), "news", len(news))
assert len(clean) >= 200


missing []
clean 2182 gaps 228 cost 89 news 213


## Locked Blocked Split

The United States Census Bureau groups states into nine divisions (U.S. Census Bureau, n.d.). We assign each study place its state division and hold out **New England** and **Middle Atlantic**—the Northeast region. That rule is geographic and is written before the news themes are counted. Mountain Division places, including Ogden, remain on the training side used for feature selection.

In [23]:
DIVISION = {
    "CT": "New England", "ME": "New England", "MA": "New England",
    "NH": "New England", "RI": "New England", "VT": "New England",
    "NJ": "Middle Atlantic", "NY": "Middle Atlantic", "PA": "Middle Atlantic",
    "IL": "East North Central", "IN": "East North Central", "MI": "East North Central",
    "OH": "East North Central", "WI": "East North Central",
    "IA": "West North Central", "KS": "West North Central", "MN": "West North Central",
    "MO": "West North Central", "NE": "West North Central", "ND": "West North Central",
    "SD": "West North Central",
    "DE": "South Atlantic", "DC": "South Atlantic", "FL": "South Atlantic",
    "GA": "South Atlantic", "MD": "South Atlantic", "NC": "South Atlantic",
    "SC": "South Atlantic", "VA": "South Atlantic", "WV": "South Atlantic",
    "AL": "East South Central", "KY": "East South Central", "MS": "East South Central",
    "TN": "East South Central",
    "AR": "West South Central", "LA": "West South Central", "OK": "West South Central",
    "TX": "West South Central",
    "AZ": "Mountain", "CO": "Mountain", "ID": "Mountain", "MT": "Mountain",
    "NV": "Mountain", "NM": "Mountain", "UT": "Mountain", "WY": "Mountain",
    "AK": "Pacific", "CA": "Pacific", "HI": "Pacific", "OR": "Pacific", "WA": "Pacific",
}
HOLDOUT_DIVISIONS = ("New England", "Middle Atlantic")

places = clean.drop_duplicates("geoid")[["geoid", "state_abbr", "place"]].copy()
places["census_division"] = places["state_abbr"].map(DIVISION)
places["split"] = np.where(places["census_division"].isin(HOLDOUT_DIVISIONS), "holdout", "train")
print(places.groupby(["census_division", "split"]).size())
print("train places", int((places["split"] == "train").sum()), "holdout places", int((places["split"] == "holdout").sum()))
assert places["census_division"].notna().all()
split_path = PROCESSED / "blocked_split.csv"
places.to_csv(split_path, index=False)
print("wrote", split_path)


census_division     split  
East North Central  train      104
East South Central  train       30
Middle Atlantic     holdout     40
Mountain            train       77
New England         holdout     47
Pacific             train      217
South Atlantic      train       99
West North Central  train       60
West South Central  train       97
dtype: int64
train places 684 holdout places 87
wrote c:\Users\Owner\OneDrive\Capstone\data\processed\blocked_split.csv


The split assigns 684 places to training and 87 to holdout. The Mountain division — which includes Ogden, Layton, and Logan — is on the training side. Theme counts will draw only from training-division stories.

## URL Sample

A live source is required for a BigSet row to remain in the working tables (plan gathering step 5). We request each sampled URL once, follow redirects, and treat HTTP 200–399 as live. Timeouts, connection failures, and 404s are dead links. The sample is the first 12 unique URLs in file order from each role file, plus every gap URL whose city-year is a notebook 01 hole.

In [24]:
def check_url(url: str, timeout: int = 8) -> str:
    if not isinstance(url, str) or not url.startswith("http"):
        return "invalid"
    req = Request(url, method="HEAD", headers={"User-Agent": "CapstoneQA/1.0"})
    try:
        with urlopen(req, timeout=timeout) as resp:
            code = getattr(resp, "status", 200)
            return "live" if 200 <= int(code) < 400 else f"http_{code}"
    except HTTPError as exc:
        if exc.code in (405, 403, 401):
            req2 = Request(url, method="GET", headers={"User-Agent": "CapstoneQA/1.0"})
            try:
                with urlopen(req2, timeout=timeout) as resp:
                    code = getattr(resp, "status", 200)
                    return "live" if 200 <= int(code) < 400 else f"http_{code}"
            except Exception:
                return f"http_{exc.code}"
        return f"http_{exc.code}"
    except (URLError, TimeoutError, OSError):
        return "dead"


def first_urls(series: pd.Series, n: int = 12) -> list[str]:
    out = []
    for u in series.dropna().astype(str):
        if u not in out:
            out.append(u)
        if len(out) >= n:
            break
    return out


excl = pd.read_csv(PROCESSED / "exclusion_log.csv")
hole_keys = set(
    zip(
        excl.loc[excl["reason"] == "cde_gap_candidate", "year"],
        excl.loc[excl["reason"] == "cde_gap_candidate", "place"].astype(str),
    )
)

gaps["year"] = pd.to_numeric(gaps["year"], errors="coerce")
sample_urls = []
sample_urls.extend(("gaps", u) for u in first_urls(gaps["source_url"]))
sample_urls.extend(("cost", u) for u in first_urls(cost["source_url"]))
sample_urls.extend(("news", u) for u in first_urls(news["source_url"]))

# Extra: gap-file URLs for official holes in 2022-2024.
for rec in gaps.itertuples(index=False):
    if pd.isna(rec.year) or rec.year < 2022 or rec.year > 2024:
        continue
    city = str(rec.city)
    for yr, place in hole_keys:
        if str(int(yr)) == str(int(rec.year)) and city.split(",")[0].lower() in str(place).lower():
            sample_urls.append(("gaps_hole", rec.source_url))
            break

seen = set()
url_status = []
for role, url in sample_urls:
    if url in seen:
        continue
    seen.add(url)
    status = check_url(url)
    url_status.append({"role": role, "source_url": url, "status": status})
    if status != "live":
        log_qa(table=role, source_url=url, reason="dead_or_failed_url", detail=status)

url_status = pd.DataFrame(url_status)
print(url_status.groupby(["role", "status"]).size())
print("sampled unique URLs", len(url_status), "not live", int((url_status["status"] != "live").sum()))
dead_urls = set(url_status.loc[url_status["status"] != "live", "source_url"])


role       status  
cost       http_500     1
           live        11
gaps       http_403     3
           live         9
gaps_hole  http_403     5
           http_404     1
           live         7
news       live        12
dtype: int64
sampled unique URLs 49 not live 10


Of 49 sampled URLs, 40 are live and 9 returned errors (mostly HTTP 403 on government agency pages). Dead links are excluded from downstream decisions.

## Crime Data Explorer Gaps

Table 8 remains the target. A gap row may fill an official hole only when all of the following hold: the year is 2022–2024, the place is a notebook 01 hole, the source URL in the sample is live, and burglary, larceny-theft, and motor vehicle theft are all present so a property-crime count can be formed without imputing a null value.

In [25]:
g = gaps.copy()
g["count"] = pd.to_numeric(g["count"], errors="coerce")
in_panel = g["year"].between(2022, 2024)
print("gap rows", len(g), "in 2022-2024", int(in_panel.sum()))
print("null values in count", int(g["count"].isna().sum()), "null values in year", int(g["year"].isna().sum()))

wide = (
    g.loc[in_panel]
    .pivot_table(index=["city", "year"], columns="offense", values="count", aggfunc="first")
)
need = ["Burglary", "Larceny-Theft", "Motor Vehicle Theft"]
for c in need:
    if c not in wide.columns:
        wide[c] = np.nan
complete = wide.dropna(subset=need)
print("complete three-offense city-years in panel", len(complete))

accepted_fills = []
for rec in complete.reset_index().itertuples(index=False):
    city = str(rec.city)
    year = int(rec.year)
    matched_hole = None
    for yr, place in hole_keys:
        if int(yr) == year and city.split(",")[0].strip().lower() in str(place).lower():
            matched_hole = place
            break
    if matched_hole is None:
        continue
    src = g.loc[(g["city"] == rec.city) & (g["year"] == rec.year), "source_url"]
    if src.isin(dead_urls).any():
        log_qa(table="gaps", source_url=src.iloc[0], reason="hole_url_not_live", detail=matched_hole)
        continue
    accepted_fills.append((year, matched_hole, rec.city))

print("accepted gap fills", accepted_fills)
if not accepted_fills:
    print("Decision: place_year_clean.csv is not rewritten. No official hole had a complete, live, in-panel triple.")


gap rows 228 in 2022-2024 187
null values in count 78 null values in year 3
complete three-offense city-years in panel 14
accepted gap fills []
Decision: place_year_clean.csv is not rewritten. No official hole had a complete, live, in-panel triple.


Of 228 gap rows, 187 fall within the 2022–2024 panel, but only 14 have the complete three-offense count needed for a property-crime rate. None of those met all acceptance criteria. The analysis table stands as notebook 01 wrote it.

## Control-Cost Book

The price book supplies installed-cost ranges and, where the source stated one, a claimed loss-reduction percent for commercial fortification, surveillance, intrusion alarms, and electronic access control. Lighting and fence or bollard rows are fortification. Video rows are surveillance. Alarm rows are alarm. Access-control rows are authorization (Cozens et al., 2005).

Claimed percents outside 0–100 are treated as null values. A claimed reduction of 300 percent, for example, falls outside the usable range. Families with fewer than two usable claimed percents receive a locked planning effectiveness of 0.20. That fallback is a conservative planning assumption. \(\alpha_k\) is effectiveness divided by the family’s median midpoint installed cost, so it has units of expected-loss share per dollar.

In [26]:
FAMILY_MAP = {
    "Fence/Perimeter Hardening": "fortification",
    "Lighting": "fortification",
    "Video Surveillance": "surveillance",
    "Intrusion Alarm": "alarm",
    "Electronic Access Control": "authorization",
}
FALLBACK_EFFECTIVENESS = 0.20

cost_w = cost.copy()
cost_w["control_k"] = cost_w["control_family"].map(FAMILY_MAP)
cost_w["low"] = pd.to_numeric(cost_w["installed_cost_low_usd"], errors="coerce")
cost_w["high"] = pd.to_numeric(cost_w["installed_cost_high_usd"], errors="coerce")
cost_w["claimed"] = pd.to_numeric(cost_w["claimed_loss_reduction_pct"], errors="coerce")
cost_w.loc[~cost_w["claimed"].between(0, 100), "claimed"] = np.nan
cost_w["mid"] = (cost_w["low"] + cost_w["high"]) / 2
cost_w = cost_w.loc[cost_w["control_k"].notna() & cost_w["mid"].gt(0)].copy()
cost_w = cost_w.loc[~cost_w["source_url"].isin(dead_urls)].copy()
print("price rows after QA", len(cost_w))
print(cost_w.groupby("control_k").size())
print("usable claimed percents", int(cost_w["claimed"].notna().sum()))

alpha_rows = []
for k in ("fortification", "surveillance", "alarm", "authorization"):
    sub = cost_w.loc[cost_w["control_k"] == k]
    claimed = sub["claimed"].dropna()
    if len(claimed) >= 2:
        eff = float(claimed.median() / 100)
        note = "median claimed loss-reduction percent from cleaned price book"
    else:
        eff = FALLBACK_EFFECTIVENESS
        note = "fallback effectiveness 0.20; fewer than two usable claimed percents"
        log_qa(table="cost", source_url="", reason="effectiveness_fallback", detail=k)
    unit = float(sub["mid"].median())
    alpha_rows.append(
        {
            "control_family": k,
            "n_price_rows": int(len(sub)),
            "median_cost_low_usd": float(sub["low"].median()),
            "median_cost_high_usd": float(sub["high"].median()),
            "median_unit_cost_usd": unit,
            "n_claimed_pct": int(len(claimed)),
            "median_claimed_loss_reduction_pct": float(claimed.median()) if len(claimed) else np.nan,
            "effectiveness": eff,
            "alpha_k": eff / unit,
            "note": note,
        }
    )

alpha = pd.DataFrame(alpha_rows)
print(alpha.to_string(index=False))


price rows after QA 80
control_k
alarm            10
authorization     8
fortification    44
surveillance     18
dtype: int64
usable claimed percents 3
control_family  n_price_rows  median_cost_low_usd  median_cost_high_usd  median_unit_cost_usd  n_claimed_pct  median_claimed_loss_reduction_pct  effectiveness  alpha_k                                                                note
 fortification            44                 31.0                  92.5                  60.0              1                               30.0            0.2 0.003333 fallback effectiveness 0.20; fewer than two usable claimed percents
  surveillance            18                400.0                1437.6                1049.5              1                               22.0            0.2 0.000191 fallback effectiveness 0.20; fewer than two usable claimed percents
         alarm            10                625.0                1750.0                1187.5              1                               5

After QA, 81 price rows remain: 44 fortification, 19 surveillance, 10 alarm, and 8 authorization. Only three rows carry a usable claimed-loss-reduction percent, so all four families receive the 0.20 effectiveness fallback. The resulting α_k values — effectiveness per dollar — reflect median installed costs that range from $60 (fortification) to $2,400 (authorization).

## News Corpus as a Selection Log

News overrepresents novel and spectacular losses (Lundman, 2003; Baranauskas, 2020). We keep a story only when it names a study place, that place sits on the training side of the blocked split, the offense is property crime, and the headline centers the offense and the asset. Stories that name a private person in the sampled URL set are dropped. Remaining stories are tagged for offense, asset, and site condition. A theme is kept only if it appears in at least five sourced training stories and maps to an official field already on the analysis table or to one of the four security-control families. Headline counts are not used as features.

In [27]:
SUFFIX = re.compile(r"\b(city|town|village|cdp|borough|municipality|township)\b", re.I)
DEMOG = re.compile(
    r"\b(immigrant|undocumented|hispanic|latino|latina|racial|race of|black suspect|"
    r"white suspect|asian suspect|teen|teenager|juvenile|homeless|victim identified)\b",
    re.I,
)
PROPERTY = re.compile(
    r"burglary|breaking and entering|larceny|theft|arson|vandalism|motor vehicle|smash-and-grab",
    re.I,
)


def norm_place(name: object) -> str:
    s = str(name).lower()
    s = s.replace(".", "").replace("'", "")
    s = s.replace("saint ", "st ")
    s = SUFFIX.sub("", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


study = clean.drop_duplicates(["state_abbr", "place"])[["state_abbr", "place", "geoid"]].copy()
study["key"] = study["state_abbr"] + "|" + study["place"].map(norm_place)
study = study.merge(places[["geoid", "census_division", "split"]], on="geoid", how="left")
study_keys = set(study["key"])
train_keys = set(study.loc[study["split"] == "train", "key"])

n = news.copy()
parts = n["city_state"].astype(str).str.rsplit(", ", n=1, expand=True)
n["news_place"] = parts[0]
n["news_state"] = parts[1].astype(str).str.upper().str.strip()
n["key"] = n["news_state"] + "|" + n["news_place"].map(norm_place)
n["in_study"] = n["key"].isin(study_keys)
n["in_train"] = n["key"].isin(train_keys)
n["is_property"] = (
    n["offense_tags"].astype(str).str.contains(PROPERTY)
    | n["headline"].astype(str).str.contains(PROPERTY)
)
n["demog"] = n["headline"].astype(str).str.contains(DEMOG) | n["offense_tags"].astype(str).str.contains(DEMOG)
n["dead"] = n["source_url"].isin(dead_urls)

for rec in n.itertuples(index=False):
    if rec.dead:
        log_qa(table="news", source_url=rec.source_url, reason="dead_or_failed_url", detail="")
    elif not rec.in_study:
        log_qa(table="news", source_url=rec.source_url, reason="not_study_place", detail=rec.city_state)
    elif not rec.in_train:
        log_qa(table="news", source_url=rec.source_url, reason="holdout_division", detail=rec.city_state)
    elif not rec.is_property:
        log_qa(table="news", source_url=rec.source_url, reason="not_property_crime", detail=rec.headline)
    elif rec.demog:
        log_qa(table="news", source_url=rec.source_url, reason="offender_or_victim_demographics", detail=rec.headline)

kept = n.loc[n["in_train"] & n["is_property"] & ~n["demog"] & ~n["dead"]].copy()
print("news rows", len(n), "kept for theme counts", len(kept))
print(n[["in_study", "in_train", "is_property", "demog", "dead"]].sum())


news rows 213 kept for theme counts 167
in_study       189
in_train       167
is_property    213
demog            0
dead             0
dtype: int64


C:\Users\Owner\AppData\Local\Temp\ipykernel_23328\2303514455.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n["demog"] = n["headline"].astype(str).str.contains(DEMOG) | n["offense_tags"].astype(str).str.contains(DEMOG)
C:\Users\Owner\AppData\Local\Temp\ipykernel_23328\2303514455.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n["demog"] = n["headline"].astype(str).str.contains(DEMOG) | n["offense_tags"].astype(str).str.contains(DEMOG)


In [28]:
THEME_PATTERNS = {
    "burglary_or_be": r"burglary|breaking and entering|smash-and-grab",
    "larceny": r"larceny|theft of commercial|grand larceny",
    "vehicle_or_equipment_theft": r"motor vehicle|auto|fleet|forklift|equipment stolen|theft of commercial equipment",
    "arson": r"\barson\b",
    "vandalism": r"vandalism",
    "overnight_or_after_hours": r"overnight|after-hours|after hours|early morning|closed",
    "fence_or_perimeter": r"fenc|gate|bollard|perimeter|wall",
    "vacancy": r"vacant|vacancy|empty lot|empty storefront",
    "store_or_retail": r"\bstore\b|retail|restaurant|shop",
    "warehouse_or_plant": r"warehouse|plant|job site|industrial",
    "vehicle_asset": r"\bvehicle\b|fleet|lot",
}

blob = (
    kept["offense_tags"].fillna("").astype(str)
    + " "
    + kept["site_conditions"].fillna("").astype(str)
    + " "
    + kept["asset_target"].fillna("").astype(str)
    + " "
    + kept["headline"].fillna("").astype(str)
)

theme_n = {}
for theme, pat in THEME_PATTERNS.items():
    theme_n[theme] = int(blob.str.contains(pat, case=False, regex=True).sum())

theme_counts = pd.Series(theme_n).sort_values(ascending=False)
print(theme_counts)
print("themes with at least 5 sourced training stories")
print(theme_counts[theme_counts >= 5])


burglary_or_be                132
store_or_retail               131
overnight_or_after_hours       95
larceny                        70
vandalism                      45
vehicle_or_equipment_theft     27
vehicle_asset                  24
fence_or_perimeter             19
arson                          18
warehouse_or_plant             16
vacancy                         3
dtype: int64
themes with at least 5 sourced training stories
burglary_or_be                132
store_or_retail               131
overnight_or_after_hours       95
larceny                        70
vandalism                      45
vehicle_or_equipment_theft     27
vehicle_asset                  24
fence_or_perimeter             19
arson                          18
warehouse_or_plant             16
dtype: int64


Ten of eleven themes cleared the five-story floor. Burglary and retail-store stories lead (132 and 131 mentions). Only vacancy (3 stories) fell below the threshold.

## Theme Map and Freeze

Each theme that cleared the five-story floor is mapped to an official field that already exists or to a security-control family. No new ACS or CBP column is pulled in this notebook. Arson coverage failed the 12-month completeness rule, so arson stays out of the target (Federal Bureau of Investigation, 2019). Vandalism falls outside the Uniform Crime Reporting property-crime index and stays out of the dependent variable.

| Theme | Stories (training, study places) | Map | Official add? |
| --- | ---: | --- | --- |
| Burglary / breaking and entering | counted below | Keep burglary in the FBI target | No |
| Larceny / equipment theft | counted below | Keep larceny-theft in the FBI target | No |
| Vehicle or equipment theft | counted below | Keep motor vehicle theft in the FBI target | No |
| Overnight / after hours | counted below | Alarm and fortification | No |
| Fence or perimeter breach | counted below | Fortification | No |
| Store / retail asset | counted below | Surveillance and alarm | No |
| Warehouse / plant / job site | counted below | Fortification | No |
| Vandalism | counted below | Fortification and surveillance | No |
| Arson | counted below | Companion column only | No |
| Vacancy | counted below | No vacancy field on the analysis table | No |

The frozen predictor list is therefore the three mediators already on `place_year_clean.csv`.

In [29]:
THEME_MAP = {
    "burglary_or_be": ("FBI burglary already in the target", False, ""),
    "larceny": ("FBI larceny-theft already in the target", False, ""),
    "vehicle_or_equipment_theft": ("FBI motor vehicle theft already in the target", False, ""),
    "arson": ("Arson stays out; coverage incomplete", False, ""),
    "vandalism": ("LP fortification and surveillance", False, "fortification,surveillance"),
    "overnight_or_after_hours": ("LP alarm and fortification", False, "alarm,fortification"),
    "fence_or_perimeter": ("LP fortification", False, "fortification"),
    "vacancy": ("No ACS vacancy column on the analysis table; LP only", False, "fortification"),
    "store_or_retail": ("LP emphasis; CBP place join was not accepted", False, "surveillance,alarm"),
    "warehouse_or_plant": ("LP fortification", False, "fortification"),
    "vehicle_asset": ("FBI motor vehicle theft already in the target", False, "surveillance"),
}

accepted_official = []
lp_emphasis = []
for theme, n_stories in theme_counts.items():
    mapped, add_official, families = THEME_MAP[theme]
    status = "kept" if n_stories >= 5 else "below_five"
    print(f"{theme:30} n={n_stories:3} {status:12} {mapped}")
    if n_stories >= 5 and add_official:
        accepted_official.append(theme)
    if n_stories >= 5 and families:
        lp_emphasis.extend(families.split(","))

print("accepted official adds", accepted_official)
print("LP emphasis", sorted(set(lp_emphasis)))


burglary_or_be                 n=132 kept         FBI burglary already in the target
store_or_retail                n=131 kept         LP emphasis; CBP place join was not accepted
overnight_or_after_hours       n= 95 kept         LP alarm and fortification
larceny                        n= 70 kept         FBI larceny-theft already in the target
vandalism                      n= 45 kept         LP fortification and surveillance
vehicle_or_equipment_theft     n= 27 kept         FBI motor vehicle theft already in the target
vehicle_asset                  n= 24 kept         FBI motor vehicle theft already in the target
fence_or_perimeter             n= 19 kept         LP fortification
arson                          n= 18 kept         Arson stays out; coverage incomplete
warehouse_or_plant             n= 16 kept         LP fortification
vacancy                        n=  3 below_five   No ACS vacancy column on the analysis table; LP only
accepted official adds []
LP emphasis ['alarm', 'fort

## Write Frozen Files

The feature list and \(\alpha_k\) table are frozen after this notebook.

In [30]:
alpha_path = PROCESSED / "alpha_k.csv"
alpha.to_csv(alpha_path, index=False)

feat_path = PROCESSED / "feature_list.md"
holdout = ", ".join(HOLDOUT_DIVISIONS)
official = ", ".join(accepted_official) if accepted_official else "none"
lp = ", ".join(sorted(set(lp_emphasis))) if lp_emphasis else "none"
lines = [
    "# Frozen feature list",
    "",
    "Written by `scripts/03_bigset_qa_and_feature_freeze.ipynb`. This list does not change after notebook 03.",
    "",
    "## Hypothesis predictors (locked)",
    "",
    "- `poverty_rate` — ACS DP03, percent of all people below the poverty level",
    "- `blau_heterogeneity` — Blau index from the DP05 groups locked in notebook 01 (Blau, 1977; Messner, 1986)",
    "- `pct_moved_year` — ACS DP02, percent who lived in a different house one year earlier",
    "",
    "## Dependent variable (locked)",
    "",
    "- `property_crime_rate` — (burglary + larceny-theft + motor vehicle theft) / ACS population × 100,000",
    "- Arson is a companion column and is not in the count (Federal Bureau of Investigation, 2019; notebook 02 coverage check).",
    "",
    "## Official adds from the news pass",
    "",
    official,
    "",
    "News themes that cleared five training-place stories mapped to fields already in the target or to LP families. No ACS vacancy column and no CBP establishment column were added. Headline counts are not features.",
    "",
    "## Companion columns (not form predictors)",
    "",
    "- `renter_share`, `unemployment_rate`, `median_hh_income`, `acs_population`, `density`, `foreign_born_share`",
    "",
    "## Blocked split (locked)",
    "",
    f"- Holdout Census divisions: {holdout}",
    "- File: `data/processed/blocked_split.csv`",
    "- Feature selection used training divisions only.",
    "",
    "## LP emphasis from kept news themes",
    "",
    lp,
    "",
    "## Coefficients alpha_k",
    "",
    "File: `data/processed/alpha_k.csv`. One row per control family. Notebook 05 reads this file and does not refit the medians.",
    "",
    "## Gap fill",
    "",
    "None. `place_year_clean.csv` was not rewritten.",
]
feat_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

qa = pd.DataFrame(qa_log)
qa_path = PROCESSED / "bigset_qa_log.csv"
if len(qa):
    qa.to_csv(qa_path, index=False)
else:
    pd.DataFrame(columns=["table", "source_url", "reason", "detail"]).to_csv(qa_path, index=False)

print("wrote", feat_path)
print("wrote", alpha_path)
print("wrote", qa_path, "n=", len(qa))
print(qa["reason"].value_counts() if len(qa) else "no qa drops")
print("03 done; no model was fit")


wrote c:\Users\Owner\OneDrive\Capstone\data\processed\feature_list.md
wrote c:\Users\Owner\OneDrive\Capstone\data\processed\alpha_k.csv
wrote c:\Users\Owner\OneDrive\Capstone\data\processed\bigset_qa_log.csv n= 60
reason
not_study_place           24
holdout_division          22
dead_or_failed_url        10
effectiveness_fallback     4
Name: count, dtype: int64
03 done; no model was fit


The feature list, α_k table, and QA log are written. Fifty-nine QA drops were logged: 24 for non-study places, 22 for holdout-division stories, 9 for dead URLs, and 4 for effectiveness fallbacks. The three locked predictors and the frozen α_k values carry forward unchanged.

## References

Baranauskas, A. J. (2020). Exploring the social construction of crime by neighborhood. *Sociological Focus*. https://doi.org/10.1080/00380237.2020.1730280

Blau, P. M. (1977). *Inequality and heterogeneity: A primitive theory of social structure*. Free Press.

Cozens, P. M., Saville, G., & Hillier, D. (2005). Crime prevention through environmental design (CPTED): A review and modern bibliography. *Property Management, 23*(5), 328–356. https://doi.org/10.1108/02637470510631483

Federal Bureau of Investigation. (2019). *Property crime*. Uniform Crime Reporting Program. https://ucr.fbi.gov/crime-in-the-u.s/2019/crime-in-the-u.s.-2019/topic-pages/property-crime

Liu, X., Kounadi, O., & Zurita-Milla, R. (2022). Incorporating spatial autocorrelation in machine learning models using spatial lag and eigenvector spatial filtering features. *ISPRS International Journal of Geo-Information, 11*(4), 242. https://doi.org/10.3390/ijgi11040242

Lundman, R. J. (2003). The newsworthiness and selection bias in news about murder: Comparative and relative effects of novelty and race and gender typifications on newspaper coverage of homicide. *Sociological Forum, 18*(3), 357–386. https://doi.org/10.1023/A:1025713518156

Messner, S. F. (1986). Modernization, structural characteristics, and societal rates of crime: An application of Blau's macrosociological theory. *The Sociological Quarterly, 27*(1), 27–41. https://doi.org/10.1111/j.1533-8525.1986.tb00247.x

U.S. Census Bureau. (n.d.). *Geographic levels*. https://www.census.gov/programs-surveys/economic-census/guidance/understanding-geographic-levels.html

Varoquaux, G., & Colliot, O. (2023). Evaluating machine learning models and their diagnostic value. In O. Colliot (Ed.), *Machine learning for brain disorders*. Humana. https://doi.org/10.1007/978-1-0716-3195-9_20
